# Demo: Linear and logistic regression

## Instituto Superior Técnico, University of Lisbon

**Description**: interactive and visual demo of linear regression and logistic regression.

Author: Rodrigo Ventura (<rodrigo.ventura@tecnico.ulisboa.pt>)

In [ ]:
import numpy as np
from sklearn.linear_model import *
from sklearn.metrics import *

%matplotlib widget
import ipywidgets as widgets
import matplotlib.pyplot as plt

## Linear regression

**Instructions**: click with the mouse on the plot to add points; the linear regression is shown in blue (requires at least 2 points).

In [ ]:
plt.ioff()
fig, ax = plt.subplots()
fig.canvas.header_visible = False
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
data, = ax.plot([], [], 'r.', label="Data")
line, = ax.plot([], [], 'b-', label="Model")
xrange = np.linspace(-1, 1, 100)
dx, dy = [], []

out = widgets.Output()
board = widgets.VBox([fig.canvas, out])
display(board)

# This function performs the linear regression and computes the MSE and MAE metrics
def linear_regression(data_x, data_y):
    X = np.array(data_x)[:,None]
    y = np.array(data_y)
    reg = LinearRegression()
    reg.fit(X, y)
    yp = reg.predict(X)
    mse = mean_squared_error(y, yp)
    mae = mean_absolute_error(y, yp)
    return (reg, mse, mae)

# This function is called at every mouse click
def on_click(event):
    if event.inaxes is not None:
        dx.append(event.xdata)
        dy.append(event.ydata)
        data.set_data(dx, dy)
        if len(dx)>1:
            # If we have at least 2 points, performs linear regression and shows the result
            (reg, mse, mae) = linear_regression(dx, dy)
            rmse = np.sqrt(mse)
            yvalues = reg.predict(xrange[:,None])
            line.set_data(xrange, yvalues)
            out.clear_output(wait=True)
            with out:
                print(f"{len(dx)} points:  RMSE = {rmse:.4f},  MAE = {mae:.4f}")
        fig.canvas.draw_idle()

fig.canvas.mpl_connect('button_press_event', on_click)

## Logistic regression

**Instructions**:
- click with the mouse on the plot to add points of one class, and shift-click for the other class
- background will show the model output (requires at least 1 point in each class)

In [ ]:
plt.ioff()
fig, ax = plt.subplots()
fig.canvas.header_visible = False
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
(Xgrid, Ygrid) = np.meshgrid(np.linspace(-1, 1, 100), np.linspace(-1, 1, 100))
XYgrid = np.vstack([Xgrid.flatten(),Ygrid.flatten()]).T
img = plt.imshow(np.zeros(Xgrid.shape), origin='lower', extent=[-1,1,-1,1], alpha=0.5)
img.set_clim(vmin=0, vmax=1)
fig.colorbar(img, ax=ax)
data0, = ax.plot([], [], 'r.', label="Data")
data1, = ax.plot([], [], 'g.', label="Data")
dx0, dy0 = [], []
dx1, dy1 = [], []

out = widgets.Output()
board = widgets.VBox([fig.canvas, out])
display(board)

# This function performs the logistic regression and computes the accuracy over the train set
def logistic_regression(data0_x, data0_y, data1_x, data1_y):
    X = np.vstack([[data0_x+data1_x], [data0_y+data1_y]]).T
    y = np.array( len(data0_x)*[0] + len(data1_y)*[1] )
    reg = LogisticRegression(C=np.inf)  # C=np.inf means no regularization
    reg.fit(X, y)
    yp = reg.predict(X)
    acc = np.sum(y==yp)/len(y) 
    return (reg, acc)

# This function is called at every mouse click
def on_click(event):
    global reg
    if event.inaxes is not None:
        if event.key is not None:
            # Input of a class 1 point (shift-click, or any other modifier)
            dx1.append(event.xdata)
            dy1.append(event.ydata)
            data1.set_data(dx1, dy1)
        else:
            # Input of a class 0 point
            dx0.append(event.xdata)
            dy0.append(event.ydata)
            data0.set_data(dx0, dy0)
        if len(dx0)>0 and len(dx1)>0:
            # If we have at least 1 point of each class, perform logistic regression
            (reg, acc) = logistic_regression(dx0, dy0, dx1, dy1)
            # Computes model output over a grid for visualization
            proba = reg.predict_proba(XYgrid)
            Zgrid = proba[:,0].reshape(Xgrid.shape)
            img.set_data(Zgrid)
            out.clear_output(wait=True)
            with out:
                print(f"{len(dx0)} + {len(dx1)} points:  Accuracy = {100*acc:.2f}%")
        fig.canvas.draw_idle()

fig.canvas.mpl_connect('button_press_event', on_click)

The next cell shows the logistic regression output in a 3D plot.

In [ ]:
plt.ion()
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
out = reg.predict_proba(XYgrid)
Zgrid = out[:,0].reshape(Xgrid.shape)
surf = ax.plot_surface(Xgrid, Ygrid, Zgrid, cmap='viridis', edgecolor='none')
plt.show()